# Cinema Sales Analysis

In [37]:
# ======================================================================
# Cinema Sales Analysis
# ======================================================================
# Author   : Amol Bajpai
# Date     : 20 November 2026
# Purpose  : Clean cinema transaction data and analyse tickets,
#            concessions, occupancy, showtimes, genres and group sizes
# Input    : "15_cinema_sales.csv" (upload when asked)
# Output   : printed tables and "cleaned_cinema_sales.csv"
# Requires : pandas, numpy
# ======================================================================

In [21]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


def section(title):
    """Print a clearly separated section header."""
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70 + "\n")

## 1. Load and inspect

In [22]:
section("1. LOAD AND INSPECT")
df = pd.read_csv("15_cinema_sales.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns \n")
print("Column types:\n")
print(df.dtypes)
print("\nFirst 5 rows:\n")
print(df.head())
print("\nSummary statistics:\n")
print(df.describe(include="all"))


  1. LOAD AND INSPECT

Shape: 1525 rows x 11 columns 

Column types:

Transaction_ID       object
Movie_Title          object
Genre                object
Screen_Format        object
Show_Date            object
Showtime_Slot        object
Ticket_Quantity       int64
Ticket_Price        float64
Ticket_Revenue      float64
Concession_Spend    float64
Occupancy_Pct       float64
dtype: object

First 5 rows:

  Transaction_ID        Movie_Title   Genre         Screen_Format   Show_Date Showtime_Slot  Ticket_Quantity  \
0       TX100294      Echoes of War   Drama           Standard 2D  2026-09-12       Evening               10   
1       TX101002     Midnight Manor  Horror           Standard 2D  2026-09-03       Evening                2   
2       TX101285  The Haunting Hour  Horror           Standard 2D  2026-10-17       Matinee                2   
3       TX100043      Turbo Chase 5  Action           Standard 2D  2026-10-02       Evening                8   
4       TX101105    Shadow Prot

## 2. Missing values and duplicate Transaction_IDs

In [23]:
section("2. MISSING VALUES AND DUPLICATES")
print("Missing values per column:\n")
print(df.isna().sum())
dupes = df["Transaction_ID"].duplicated().sum()
print(f"\nDuplicate Transaction_IDs: {dupes}")
df = df.drop_duplicates(subset="Transaction_ID", keep="first")
print(f"Rows after removing duplicates: {len(df)}")


  2. MISSING VALUES AND DUPLICATES

Missing values per column:

Transaction_ID       0
Movie_Title          0
Genre                0
Screen_Format        0
Show_Date            0
Showtime_Slot        0
Ticket_Quantity      0
Ticket_Price         0
Ticket_Revenue       0
Concession_Spend    94
Occupancy_Pct        0
dtype: int64

Duplicate Transaction_IDs: 25
Rows after removing duplicates: 1500


## 3. Clean Concession_Spend

In [24]:
# Missing spend most plausibly means "bought nothing" if the till only logs purchases.
section("3. CLEAN CONCESSION_SPEND")
missing_spend = df["Concession_Spend"].isna().sum()
df["Concession_Spend"] = df["Concession_Spend"].fillna(0)
print(f"Filled {missing_spend} missing Concession_Spend values with 0")


  3. CLEAN CONCESSION_SPEND

Filled 90 missing Concession_Spend values with 0


## 4. Dates and weekend flag

In [25]:
section("4. DATES AND WEEKEND FLAG")
df["Show_Date"] = pd.to_datetime(df["Show_Date"], errors="coerce")
bad_dates = df["Show_Date"].isna().sum()
print(f"Unparseable dates dropped: {bad_dates}")
df = df.dropna(subset=["Show_Date"])
df["Weekday"] = df["Show_Date"].dt.day_name()
df["Is_Weekend"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
print("\nTransactions by day type:\n")
print(df["Is_Weekend"].value_counts())


  4. DATES AND WEEKEND FLAG

Unparseable dates dropped: 0

Transactions by day type:

Is_Weekend
Weekday    836
Weekend    664
Name: count, dtype: int64


## 5. Revenue totals

In [26]:
# Adjust if Ticket_Revenue isn't a column
section("5. REVENUE TOTALS")
if "Ticket_Revenue" not in df:
    df["Ticket_Revenue"] = df["Ticket_Quantity"] * df["Ticket_Price"]
df["Total_Spend"] = df["Ticket_Revenue"] + df["Concession_Spend"]
print(df[["Ticket_Revenue", "Concession_Spend", "Total_Spend"]].sum().round(2))


  5. REVENUE TOTALS

Ticket_Revenue      67872.00
Concession_Spend    16578.36
Total_Spend         84450.36
dtype: float64


## 6. Top 10 movies by tickets sold

In [27]:
section("6. TOP 10 MOVIES BY TICKETS SOLD")
top10 = (df.groupby("Movie_Title")
           .agg(Tickets=("Ticket_Quantity", "sum"), Revenue=("Ticket_Revenue", "sum"))
           .sort_values("Tickets", ascending=False).head(10))
print(top10.round(2))


  6. TOP 10 MOVIES BY TICKETS SOLD

                     Tickets  Revenue
Movie_Title                          
Turbo Chase 5            510   6747.0
Midnight Manor           491   6837.0
Galactic Frontier        473   6410.0
Dragon Realm             414   5804.0
Tiny Paws Adventure      410   5624.0
The Last Heist           389   5062.0
Shadow Protocol          352   4569.0
Laugh Out Loud           303   4034.0
Neon City 2099           276   3630.0
Love in Lisbon           273   3698.0


## 7. Average occupancy by screen format

In [28]:
section("7. AVERAGE OCCUPANCY BY SCREEN FORMAT")
print(df.groupby("Screen_Format")["Occupancy_Pct"].mean().sort_values(ascending=False).round(2))


  7. AVERAGE OCCUPANCY BY SCREEN FORMAT

Screen_Format
IMAX                    80.96
Premium Large Format    77.30
4DX                     72.28
Standard 2D             64.28
3D                      57.63
Name: Occupancy_Pct, dtype: float64


## 8. Weekday vs weekend

In [29]:
section("8. WEEKDAY VS WEEKEND")
print(df.groupby("Is_Weekend")[["Ticket_Quantity", "Ticket_Revenue", "Concession_Spend"]]
        .agg(["sum", "mean"]).round(2))
days = df.groupby("Is_Weekend")["Show_Date"].nunique()
print("\nDistinct dates per group:\n")
print(days)
print("\nAverage revenue per day:\n")
print((df.groupby("Is_Weekend")["Ticket_Revenue"].sum() / days).round(2))


  8. WEEKDAY VS WEEKEND

           Ticket_Quantity       Ticket_Revenue        Concession_Spend       
                       sum  mean            sum   mean              sum   mean
Is_Weekend                                                                    
Weekday               2862  3.42        37256.0  44.56          8801.99  10.53
Weekend               2206  3.32        30616.0  46.11          7776.37  11.71

Distinct dates per group:

Is_Weekend
Weekday    44
Weekend    17
Name: Show_Date, dtype: int64

Average revenue per day:

Is_Weekend
Weekday     846.73
Weekend    1800.94
dtype: float64


## 9. Showtime slots

In [30]:
section("9. SHOWTIME SLOTS")
print(df.groupby("Showtime_Slot")
        .agg(Tickets=("Ticket_Quantity", "sum"),
             Transactions=("Transaction_ID", "count"),
             Avg_Occupancy=("Occupancy_Pct", "mean")).round(2))


  9. SHOWTIME SLOTS

               Tickets  Transactions  Avg_Occupancy
Showtime_Slot                                      
Evening           2790           818          74.15
Late Night         989           294          55.32
Matinee           1289           388          61.04


## 10. Concession attach rate

In [31]:
section("10. CONCESSION ATTACH RATE")
attach = (df["Concession_Spend"] > 0).mean() * 100
print(f"Overall attach rate: {attach:.1f}%\n")
print("By day type (%):\n")
print(df.groupby("Is_Weekend")["Concession_Spend"].apply(lambda s: (s > 0).mean() * 100).round(1))


  10. CONCESSION ATTACH RATE

Overall attach rate: 56.1%

By day type (%):

Is_Weekend
Weekday    53.0
Weekend    60.1
Name: Concession_Spend, dtype: float64


## 11. Average spend by genre

In [32]:
section("11. AVERAGE SPEND BY GENRE")
print(df.groupby("Genre")[["Ticket_Revenue", "Concession_Spend"]].mean().round(2))


  11. AVERAGE SPEND BY GENRE

           Ticket_Revenue  Concession_Spend
Genre                                      
Action              45.08             12.08
Animation           42.29             13.61
Comedy              40.27             12.63
Drama               49.58             10.36
Fantasy             47.57             10.39
Horror              47.08              9.71
Romance             46.25             11.02
Sci-Fi              44.42             11.07
Thriller            46.02              7.34


## 12. Total spend statistics (NumPy)

In [33]:
section("12. TOTAL SPEND STATISTICS")
spend = df["Total_Spend"].dropna().to_numpy()
print(f"Mean   : {np.mean(spend):.2f}")
print(f"Median : {np.median(spend):.2f}")
print(f"Std Dev: {np.std(spend, ddof=1):.2f}")


  12. TOTAL SPEND STATISTICS

Mean   : 56.30
Median : 38.57
Std Dev: 53.42


## 13. Ticket quantity outliers (IQR)

In [34]:
section("13. TICKET QUANTITY OUTLIERS (IQR)")
q1, q3 = np.percentile(df["Ticket_Quantity"], [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
large = df[df["Ticket_Quantity"] > upper]
print(f"Q1: {q1}  Q3: {q3}  IQR: {iqr}")
print(f"Upper fence: {upper}")
print(f"Large bookings: {len(large)}")
print("\nTicket quantity distribution:\n")
print(df["Ticket_Quantity"].value_counts().sort_index())


  13. TICKET QUANTITY OUTLIERS (IQR)

Q1: 2.0  Q3: 4.0  IQR: 2.0
Upper fence: 7.0
Large bookings: 137

Ticket quantity distribution:

Ticket_Quantity
1     257
2     582
3     196
4     194
5      84
6      50
8      38
10     43
12     27
15     15
20     14
Name: count, dtype: int64


## 14. Group-size bands

In [35]:
section("14. GROUP-SIZE BANDS")
df["Group_Band"] = pd.cut(df["Ticket_Quantity"], bins=[0, 1, 2, 5, np.inf],
                          labels=["Solo", "Couple", "Small Group", "Large Group"])
print(df.groupby("Group_Band", observed=True)
        .agg(Transactions=("Transaction_ID", "count"),
             Avg_Concession=("Concession_Spend", "mean")).round(2))

print("\n" + "=" * 70)
print("  ANALYSIS COMPLETE")
print("=" * 70 + "\n")


  14. GROUP-SIZE BANDS

             Transactions  Avg_Concession
Group_Band                               
Solo                  257            3.12
Couple                582            6.93
Small Group           474           12.51
Large Group           187           31.08

  ANALYSIS COMPLETE

